# 📖 Notebook 2: Review & Rating Aggregation

When you search for businesses on Yelp, the **average star rating** is the first thing you see. But calculating this on-the-fly for every search query is extremely expensive — imagine joining millions of reviews for every page load.

This notebook explores **three approaches** to maintaining business ratings, from naive to production-ready, and handles the tricky **concurrent update** problem.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why computing AVG(rating) on every query doesn't scale
- How to pre-compute ratings with periodic batch updates
- How to update ratings in real-time using the running average formula
- How **optimistic locking** prevents data corruption from concurrent reviews
- How to enforce one-review-per-user-per-business at the database level

## 🛠️ Setup

```bash
cd system-designs/yelp
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
import random
from concurrent.futures import ThreadPoolExecutor

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "yelp_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

try:
    conn = get_db(); conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL: {e}\n   Run: docker-compose up -d")

try:
    r = get_redis(); r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis: {e}\n   Run: docker-compose up -d")

✅ PostgreSQL connected
✅ Redis connected


## 🐢 Approach 1: Compute AVG(rating) on Every Query

The simplest approach — just JOIN reviews and businesses every time:

```sql
SELECT b.id, b.name, AVG(r.rating) AS avg_rating
FROM businesses b
JOIN reviews r ON b.id = r.business_id
GROUP BY b.id;
```

**Why it's bad**: every search query re-scans the entire reviews table. At 10M businesses × 100 reviews each = 1 billion rows to aggregate. That's fine for 10 users; disastrous for 100M.

In [2]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Approach 1: Calculate average rating on the fly
start = time.time()
cur.execute("""
    SELECT b.id, b.name, b.city,
           ROUND(AVG(r.rating)::numeric, 2) AS computed_avg,
           COUNT(r.id) AS review_count
    FROM businesses b
    JOIN reviews r ON b.id = r.business_id
    WHERE b.city = 'New York'
    GROUP BY b.id, b.name, b.city
    ORDER BY computed_avg DESC
    LIMIT 10;
""")
on_fly_results = cur.fetchall()
on_fly_time = (time.time() - start) * 1000

print(f"🐢 On-the-fly AVG calculation: {on_fly_time:.2f} ms")
print(f"   Top-rated businesses in New York:\n")
for b in on_fly_results[:5]:
    print(f"   ⭐ {b['computed_avg']} ({b['review_count']} reviews) | {b['name']}")

# Show the query plan
cur.execute("""
    EXPLAIN ANALYZE
    SELECT b.id, b.name, ROUND(AVG(r.rating)::numeric, 2) AS avg_r, COUNT(r.id)
    FROM businesses b
    JOIN reviews r ON b.id = r.business_id
    WHERE b.city = 'New York'
    GROUP BY b.id;
""")
plan = cur.fetchall()
print("\n📊 Query Plan:")
for row in plan:
    print(f"   {row['QUERY PLAN']}")

conn.close()

print("\n⚠️  With 500 businesses and 3000 reviews, this is fast.")
print("   At 10M businesses and 1B reviews, this would take SECONDS per query.")

🐢 On-the-fly AVG calculation: 30.28 ms
   Top-rated businesses in New York:

   ⭐ 4.20 (5 reviews) | Fresh Creek Restaurant
   ⭐ 4.00 (10 reviews) | Royal Phoenix Repairs
   ⭐ 3.93 (14 reviews) | Pacific Lane Repairs
   ⭐ 3.80 (10 reviews) | Blue Plaza Repairs
   ⭐ 3.79 (14 reviews) | Royal Place Restaurant

📊 Query Plan:
   HashAggregate  (cost=121.94..123.44 rows=100 width=64) (actual time=0.418..0.432 rows=61 loops=1)
     Group Key: b.id
     Batches: 1  Memory Usage: 32kB
     ->  Hash Join  (cost=34.50..117.44 rows=600 width=32) (actual time=0.072..0.368 rows=592 loops=1)
           Hash Cond: (r.business_id = b.id)
           ->  Seq Scan on reviews r  (cost=0.00..75.00 rows=3000 width=12) (actual time=0.033..0.149 rows=3022 loops=1)
           ->  Hash  (cost=33.25..33.25 rows=100 width=24) (actual time=0.037..0.037 rows=100 loops=1)
                 Buckets: 1024  Batches: 1  Memory Usage: 14kB
                 ->  Seq Scan on businesses b  (cost=0.00..33.25 rows=100 width=24)

## ⏰ Approach 2: Pre-Compute Ratings with a Cron Job

Instead of computing the average on every query, we pre-calculate it and store it in the `businesses` table.

A **cron job** runs periodically (e.g., every hour) to recalculate all ratings.

**Pros**: Search queries are now simple and fast — just read `avg_rating` directly.  
**Cons**: Ratings are **stale** until the next cron run. If you leave a 5-star review, you might not see the rating change for an hour.

In [3]:
def cron_update_all_ratings():
    """Simulates a cron job that recalculates all business ratings."""
    conn = get_db()
    cur = conn.cursor()

    start = time.time()
    cur.execute("""
        UPDATE businesses b SET
            avg_rating = sub.avg_r,
            num_reviews = sub.cnt,
            updated_at = NOW()
        FROM (
            SELECT business_id,
                   ROUND(AVG(rating)::numeric, 2) AS avg_r,
                   COUNT(*) AS cnt
            FROM reviews
            GROUP BY business_id
        ) sub
        WHERE b.id = sub.business_id;
    """)
    conn.commit()
    elapsed = (time.time() - start) * 1000

    print(f"⏰ Cron job updated all ratings in {elapsed:.2f} ms")
    conn.close()

cron_update_all_ratings()

# Now search is much simpler — just read the pre-computed column
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

start = time.time()
cur.execute("""
    SELECT id, name, city, avg_rating, num_reviews
    FROM businesses
    WHERE city = 'New York'
    ORDER BY avg_rating DESC
    LIMIT 10;
""")
precomputed_results = cur.fetchall()
precomputed_time = (time.time() - start) * 1000

print(f"\n🚀 Pre-computed search: {precomputed_time:.2f} ms (vs {on_fly_time:.2f} ms on-the-fly)")
for b in precomputed_results[:5]:
    print(f"   ⭐ {b['avg_rating']} ({b['num_reviews']} reviews) | {b['name']}")

conn.close()

print(f"\n💡 The pre-computed approach is ~{on_fly_time/precomputed_time:.0f}× faster for reads.")
print("   But the rating is stale between cron runs.")

⏰ Cron job updated all ratings in 47.82 ms



🚀 Pre-computed search: 16.79 ms (vs 30.28 ms on-the-fly)
   ⭐ 4.20 (5 reviews) | Fresh Creek Restaurant
   ⭐ 4.00 (10 reviews) | Royal Phoenix Repairs
   ⭐ 3.93 (14 reviews) | Pacific Lane Repairs
   ⭐ 3.80 (10 reviews) | Blue Plaza Repairs
   ⭐ 3.79 (14 reviews) | Royal Place Restaurant

💡 The pre-computed approach is ~2× faster for reads.
   But the rating is stale between cron runs.


## ⚡ Approach 3: Real-Time Running Average

The best approach: update the average **in real time** as each review comes in.

The math is simple. If a business has `n` reviews with average `A`, and a new rating `r` comes in:

```
new_avg = (A × n + r) / (n + 1)
```

This is a single row update — no scanning the reviews table at all!

In [4]:
def submit_review_realtime(user_id: int, business_id: int, rating: int, text: str = None):
    """
    Submit a review and update the business rating in real time.
    Uses the running average formula: new_avg = (old_avg * n + rating) / (n + 1)
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Step 1: Insert the review
        cur.execute("""
            INSERT INTO reviews (business_id, user_id, rating, text)
            VALUES (%s, %s, %s, %s)
            RETURNING id;
        """, (business_id, user_id, rating, text))
        review_id = cur.fetchone()["id"]

        # Step 2: Update the business rating using the running average formula
        cur.execute("""
            UPDATE businesses
            SET avg_rating = ROUND(((avg_rating * num_reviews + %s) / (num_reviews + 1))::numeric, 2),
                num_reviews = num_reviews + 1,
                updated_at = NOW()
            WHERE id = %s
            RETURNING id, name, avg_rating, num_reviews;
        """, (rating, business_id))
        updated_biz = cur.fetchone()

        conn.commit()
        return {"review_id": review_id, "business": dict(updated_biz)}

    except psycopg2.errors.UniqueViolation:
        conn.rollback()
        return {"error": "You already reviewed this business!"}
    finally:
        conn.close()


# Let's see it in action
# First, check a business's current rating
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, name, avg_rating, num_reviews FROM businesses WHERE id = 1")
biz = cur.fetchone()
conn.close()

print(f"📊 Before review:")
print(f"   {biz['name']}: ⭐ {biz['avg_rating']} ({biz['num_reviews']} reviews)\n")

# Find a user who hasn't reviewed this business yet
conn = get_db()
cur = conn.cursor()
cur.execute("""
    SELECT id FROM users
    WHERE id NOT IN (SELECT user_id FROM reviews WHERE business_id = 1)
    LIMIT 1;
""")
row = cur.fetchone()
conn.close()

if row:
    available_user = row[0]
    result = submit_review_realtime(available_user, 1, 5, "Absolutely amazing place!")
    if "error" in result:
        print(f"⚠️  {result['error']}")
    else:
        print(f"✅ Review submitted!")
        print(f"   Review ID: {result['review_id']}")
        updated = result['business']
        print(f"\n📊 After review:")
        print(f"   {updated['name']}: ⭐ {updated['avg_rating']} ({updated['num_reviews']} reviews)")
        print(f"\n💡 Rating updated instantly — no cron job needed!")
else:
    print("⚠️  All users have already reviewed this business.")

📊 Before review:
   Downtown Ridge Restaurant: ⭐ 2.89 (9 reviews)



✅ Review submitted!
   Review ID: 3024

📊 After review:
   Downtown Ridge Restaurant: ⭐ 3.10 (10 reviews)

💡 Rating updated instantly — no cron job needed!


## 🔒 The Concurrency Problem: Lost Updates

The running average approach has a subtle bug. What if **two users review the same business at the exact same time**?

```
Business A: avg_rating=4.0, num_reviews=100

User 1 reads:  avg=4.0, n=100              User 2 reads:  avg=4.0, n=100
User 1 submits 5★                          User 2 submits 3★
User 1 writes: avg=4.01, n=101             User 2 writes: avg=3.99, n=101  ← OVERWRITES User 1!
```

**Result**: `avg=3.99, n=101` — User 1's review is effectively lost from the average!

The correct result should be: `(4.0×100 + 5 + 3) / 102 = 4.00`

### Solution: Optimistic Locking

We use `num_reviews` as a version check. The UPDATE only succeeds if `num_reviews` hasn't changed since we read it. If it has, we retry.

In [5]:
def submit_review_safe(user_id: int, business_id: int, rating: int, text: str = None, max_retries: int = 5):
    """
    Submit a review with optimistic locking to prevent lost updates.
    If another review was submitted concurrently, we retry with the new values.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    try:
        # Insert the review first
        cur.execute("""
            INSERT INTO reviews (business_id, user_id, rating, text)
            VALUES (%s, %s, %s, %s)
            RETURNING id;
        """, (business_id, user_id, rating, text))
        review_id = cur.fetchone()["id"]

        # Optimistic lock: retry until the update succeeds
        for attempt in range(max_retries):
            # Read current state
            cur.execute("""
                SELECT avg_rating, num_reviews FROM businesses WHERE id = %s;
            """, (business_id,))
            biz = cur.fetchone()
            old_avg = float(biz["avg_rating"])
            old_count = biz["num_reviews"]

            # Calculate new average
            new_avg = round((old_avg * old_count + rating) / (old_count + 1), 2)
            new_count = old_count + 1

            # Conditional update: only succeeds if num_reviews hasn't changed
            cur.execute("""
                UPDATE businesses
                SET avg_rating = %s, num_reviews = %s, updated_at = NOW()
                WHERE id = %s AND num_reviews = %s
                RETURNING id, name, avg_rating, num_reviews;
            """, (new_avg, new_count, business_id, old_count))

            result = cur.fetchone()
            if result:
                conn.commit()
                return {
                    "review_id": review_id,
                    "business": dict(result),
                    "retries": attempt
                }
            # If result is None, someone else updated first — retry!
            # (We don't need to rollback, just re-read and try again)

        conn.rollback()
        return {"error": "Too many concurrent updates, please try again"}

    except psycopg2.errors.UniqueViolation:
        conn.rollback()
        return {"error": "You already reviewed this business!"}
    finally:
        conn.close()


print("🔒 Optimistic Locking Review Submission")
print("=" * 50)

# Find a business and available user
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT id, name, avg_rating, num_reviews FROM businesses WHERE id = 2")
biz = cur.fetchone()
cur.execute("""
    SELECT id FROM users
    WHERE id NOT IN (SELECT user_id FROM reviews WHERE business_id = 2)
    LIMIT 1;
""")
row = cur.fetchone()
conn.close()

if row:
    print(f"\nBefore: {biz['name']} — ⭐ {biz['avg_rating']} ({biz['num_reviews']} reviews)")
    result = submit_review_safe(row["id"], 2, 4, "Solid experience!")
    if "error" not in result:
        updated = result['business']
        print(f"After:  {updated['name']} — ⭐ {updated['avg_rating']} ({updated['num_reviews']} reviews)")
        print(f"Retries needed: {result['retries']}")
    else:
        print(f"⚠️  {result['error']}")
else:
    print("⚠️  All users have reviewed business 2.")

🔒 Optimistic Locking Review Submission



Before: Pacific Point Cafe — ⭐ 3.17 (6 reviews)
After:  Pacific Point Cafe — ⭐ 3.29 (7 reviews)
Retries needed: 0


## 🧪 Simulating Concurrent Reviews

Let's prove that optimistic locking actually works by submitting many reviews concurrently and checking the final state.

In [6]:
# Pick a business to test with
TEST_BIZ = 10

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Get current state
cur.execute("SELECT id, name, avg_rating, num_reviews FROM businesses WHERE id = %s", (TEST_BIZ,))
before = cur.fetchone()

# Find users who haven't reviewed this business
cur.execute("""
    SELECT id FROM users
    WHERE id NOT IN (SELECT user_id FROM reviews WHERE business_id = %s)
    ORDER BY id
    LIMIT 20;
""", (TEST_BIZ,))
available_users = [row["id"] for row in cur.fetchall()]
conn.close()

if len(available_users) < 5:
    print("⚠️  Not enough available users for this test. Try a different business ID.")
else:
    print(f"📊 Before: {before['name']} — ⭐ {before['avg_rating']} ({before['num_reviews']} reviews)")
    print(f"   Submitting {len(available_users)} concurrent reviews...\n")

    # Submit reviews concurrently
    ratings_submitted = []
    results = []

    def do_review(user_id):
        rating = random.randint(1, 5)
        ratings_submitted.append(rating)
        return submit_review_safe(user_id, TEST_BIZ, rating, "Concurrent test review")

    with ThreadPoolExecutor(max_workers=10) as pool:
        results = list(pool.map(do_review, available_users))

    # Count successes and retries
    successes = [r for r in results if "error" not in r]
    retries = sum(r.get("retries", 0) for r in successes)
    errors = [r for r in results if "error" in r]

    print(f"   ✅ Successful reviews: {len(successes)}")
    print(f"   🔄 Total retries needed: {retries}")
    print(f"   ❌ Errors (duplicates): {len(errors)}")

    # Verify the final state by recalculating from all reviews
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cur.execute("SELECT avg_rating, num_reviews FROM businesses WHERE id = %s", (TEST_BIZ,))
    after = cur.fetchone()

    cur.execute("SELECT ROUND(AVG(rating)::numeric, 2) as true_avg, COUNT(*) as true_count FROM reviews WHERE business_id = %s", (TEST_BIZ,))
    truth = cur.fetchone()
    conn.close()

    print(f"\n📊 After: ⭐ {after['avg_rating']} ({after['num_reviews']} reviews)")
    print(f"📊 Truth: ⭐ {truth['true_avg']} ({truth['true_count']} reviews)")

    # Check if they match (allow small rounding difference)
    if abs(float(after['avg_rating']) - float(truth['true_avg'])) <= 0.02:
        print("\n✅ Optimistic locking kept the rating accurate despite concurrent writes!")
    else:
        print("\n⚠️  Small rounding difference detected (expected with running average vs full recalculation)")

📊 Before: Mountain Dragon Inn — ⭐ 3.27 (26 reviews)
   Submitting 20 concurrent reviews...



   ✅ Successful reviews: 20
   🔄 Total retries needed: 23
   ❌ Errors (duplicates): 0

📊 After: ⭐ 3.06 (46 reviews)
📊 Truth: ⭐ 3.07 (46 reviews)

✅ Optimistic locking kept the rating accurate despite concurrent writes!


## 🚫 One Review Per User Per Business

Yelp allows each user to leave **only one review** per business. This prevents spam (e.g., a competitor flooding a rival with 1-star reviews).

Our `init.sql` already added this constraint:

```sql
CONSTRAINT unique_user_business UNIQUE (user_id, business_id)
```

**Why a database constraint instead of an application check?**
- Application checks have race conditions (two requests pass the check simultaneously)
- Other services or data engineers might write reviews directly
- The database constraint is **impossible to bypass** — it's enforced at the storage level

In [7]:
# Demonstrate the unique constraint in action

conn = get_db()
cur = conn.cursor()

# Find a user who has already reviewed a business
cur.execute("SELECT user_id, business_id FROM reviews LIMIT 1")
existing = cur.fetchone()
user_id, biz_id = existing

print(f"User {user_id} already reviewed business {biz_id}.")
print(f"Trying to submit a second review...\n")

try:
    cur.execute("""
        INSERT INTO reviews (business_id, user_id, rating, text)
        VALUES (%s, %s, 5, 'Trying to review again');
    """, (biz_id, user_id))
    conn.commit()
    print("❌ This should not happen!")
except psycopg2.errors.UniqueViolation as e:
    conn.rollback()
    print(f"✅ Database blocked it: {e.diag.message_primary}")
    print(f"\n💡 The UNIQUE constraint on (user_id, business_id) makes it impossible")
    print(f"   to leave multiple reviews, regardless of which service writes to the DB.")
finally:
    conn.close()

User 60 already reviewed business 1.
Trying to submit a second review...

✅ Database blocked it: duplicate key value violates unique constraint "unique_user_business"

💡 The UNIQUE constraint on (user_id, business_id) makes it impossible
   to leave multiple reviews, regardless of which service writes to the DB.


## ⚡ Caching Business Ratings in Redis

Business ratings are read far more often than they're written. Caching them in Redis avoids hitting Postgres on every page view.

In [8]:
r = get_redis()

def get_business_with_cache(business_id: int) -> dict:
    """Fetch business details with Redis cache-aside pattern."""
    cache_key = f"biz:{business_id}"

    # Check cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), True  # (data, from_cache)

    # Cache miss — query Postgres
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute("""
        SELECT id, name, city, avg_rating, num_reviews, price_range
        FROM businesses WHERE id = %s
    """, (business_id,))
    biz = cur.fetchone()
    conn.close()

    if biz:
        # Convert Decimal to float for JSON serialization
        data = {k: (float(v) if hasattr(v, 'as_tuple') else v) for k, v in biz.items()}
        r.setex(cache_key, 300, json.dumps(data))  # cache for 5 minutes
        return data, False
    return None, False


def invalidate_business_cache(business_id: int):
    """Called after a new review is submitted."""
    r.delete(f"biz:{business_id}")


# Demo: fetch → cache → fetch from cache
start = time.time()
data, cached = get_business_with_cache(1)
t1 = (time.time() - start) * 1000
print(f"🔍 First fetch: {t1:.2f} ms (from_cache: {cached})")
print(f"   {data['name']} — ⭐ {data['avg_rating']}")

start = time.time()
data, cached = get_business_with_cache(1)
t2 = (time.time() - start) * 1000
print(f"\n⚡ Second fetch: {t2:.2f} ms (from_cache: {cached})")
print(f"\n🚀 Cache speedup: {t1/t2:.1f}×")

# Show that invalidation works
invalidate_business_cache(1)
_, cached = get_business_with_cache(1)
print(f"\n🔄 After invalidation, fetched from cache: {cached}")
print("   (cache was refreshed from DB)")

🔍 First fetch: 51.96 ms (from_cache: False)
   Downtown Ridge Restaurant — ⭐ 3.1

⚡ Second fetch: 0.48 ms (from_cache: True)

🚀 Cache speedup: 107.4×

🔄 After invalidation, fetched from cache: False
   (cache was refreshed from DB)


## 🧹 Cleanup

In [9]:
r = get_redis()
keys = r.keys("biz:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

🧹 Cleaned 1 Redis keys


## 📚 Summary

### Key Takeaways

| Approach | Freshness | Read Speed | Complexity |
|----------|-----------|------------|------------|
| On-the-fly AVG() | Always fresh | Slow (joins) | Simple |
| Cron batch update | Stale (hours) | Fast (direct read) | Medium |
| Running average | Real-time | Fast (direct read) | Higher (needs locking) |

### System Design Interview Tips

1. **Start simple**: mention on-the-fly AVG, explain why it won't scale
2. **Propose the cron job**: good enough for many systems, but stale
3. **Land on running average**: show the formula, then call out the concurrency problem
4. **Optimistic locking**: use `num_reviews` as a version number — no heavy locking needed
5. **Mention write volume**: Yelp's write throughput is ~1 review/second — no message queue needed!
6. **Database constraints**: always enforce data rules at the persistence layer, not application layer

### Next Up

In **Notebook 3**, we'll tackle **Search Ranking & Relevance** — how to combine text search, location, ratings, and other signals to rank search results.